# 02 Praproses Pendikom


In [ ]:
from pathlib import Path
import os, csv, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT=Path(os.environ.get('CARE_KM_ROOT','/content/drive/MyDrive/REVISED'))
GROUNDTRUTH_CSV=PROJECT_ROOT/'DATA'/'compiled_pendikom_moodle_logs.csv'
OUTPUT_DIR=PROJECT_ROOT/'OUTPUT'/'PRAPROSES'/'02_Praproses_Pendikom'
print('Input:',GROUNDTRUTH_CSV)
print('Output:',OUTPUT_DIR)


## Pembacaan sumber dan split peserta


In [ ]:

# ============================================================
# Dataset 2 — PENDIKOM compiled Moodle + ILS questionnaire
# ============================================================

PENDIKOM_FEATURES = [
    'content_visit','content_stay','selfass_visit','selfass_twice_wrong',
    'exercise_visit','exercise_stay','ques_detail','ques_overview',
    'ques_facts','ques_concepts','ques_graphics','ques_text',
    'ques_interpret','ques_develop','quiz_revisions','quiz_stay_results',
    'navigation_skip','navigation_overview_visit','navigation_overview_stay',
]

PENDIKOM_THRESHOLDS = {
    'content_visit': (75,100),
    'content_stay': (50,75),
    'selfass_visit': (25,75),
    'selfass_twice_wrong': (25,50),
    'exercise_visit': (25,75),
    'exercise_stay': (50,75),
    'ques_detail': (50,75),
    'ques_overview': (50,75),
    'ques_facts': (50,75),
    'ques_concepts': (50,75),
    'ques_graphics': (50,75),
    'ques_text': (50,75),
    'ques_interpret': (50,75),
    'ques_develop': (50,75),
    'quiz_revisions': (20,50),
    'quiz_stay_results': (30,60),
    'navigation_skip': (1,2),
    'navigation_overview_visit': (10,20),
    'navigation_overview_stay': (50,75),
}

ILS_CODE_COLS = [
    'ils_active_reflective_code',
    'ils_sensing_intuitive_code',
    'ils_visual_verbal_code',
    'ils_sequential_global_code',
]


def read_compiled_pendikom(path):
    """Read normal multi-column CSV or the repository one-column wrapped CSV."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)

    with path.open('r', encoding='utf-8-sig', newline='') as fh:
        outer = list(csv.reader(fh))

    if outer and all(len(r) == 1 for r in outer):
        inner = [next(csv.reader([r[0]])) for r in outer]
        width = len(inner[0])
        bad = [i for i, r in enumerate(inner) if len(r) != width]
        if bad:
            raise RuntimeError(f'Wrapped CSV has inconsistent row widths: {bad[:10]}')
        frame = pd.DataFrame(inner[1:], columns=inner[0])
    else:
        frame = pd.read_csv(path, low_memory=False)

    frame.columns = [str(c).strip() for c in frame.columns]

    text_cols = {
        'record_group','username_anon','idnumber_anon','course_fullname',
        'ils_course_fullname','ils_active_reflective_code',
        'ils_sensing_intuitive_code','ils_visual_verbal_code',
        'ils_sequential_global_code','ils_ls_class','ils_semantic_label',
    } | {f'Q{i:02d}' for i in range(1,45)}

    for c in frame.columns:
        if c not in text_cols:
            frame[c] = pd.to_numeric(frame[c], errors='coerce')

    return frame


pendikom_all = read_compiled_pendikom(GROUNDTRUTH_CSV)

required_source_cols = {
    'has_ils','userid','courseid','total_activity','active_days',
    'content_available','content_visit_evidence_count','content_stay_seconds',
    'question_available','question_attempt_count','repeated_question_count',
    'twice_wrong_question_count','assign_available','submitted_assignment_count',
    'exercise_stay_seconds','quiz_results_stay_seconds_sum',
    'quiz_results_event_count','revised_question_attempt_count',
    'module_view_count','navigation_skip_count','overview_visit_count',
    'overview_stay_seconds','ils_ls_class', *ILS_CODE_COLS,
}
for tag in ['detail','overview','facts','concepts','graphics','text','develop']:
    required_source_cols |= {
        f'ques_{tag}_fraction_sum',
        f'ques_{tag}_attempted_count',
    }
required_source_cols |= {
    'interpret_question_score_sum',
    'ques_interpret_attempted_count',
}

missing = sorted(required_source_cols - set(pendikom_all.columns))
if missing:
    raise RuntimeError(f'PENDIKOM source missing required columns: {missing}')

pendikom_ils = pendikom_all[pendikom_all['has_ils'].eq(1)].copy()
pendikom_ils['userid'] = pd.to_numeric(pendikom_ils['userid'], errors='raise').astype(int)
pendikom_ils['courseid'] = pd.to_numeric(pendikom_ils['courseid'], errors='raise').astype(int)

if pendikom_ils['userid'].duplicated().any():
    raise RuntimeError('Duplicate userid among WITH_ILS learners.')

if len(pendikom_ils) != 461:
    raise RuntimeError(f'Expected 461 WITH_ILS learners, found {len(pendikom_ils)}.')

if pendikom_ils[ILS_CODE_COLS + ['ils_ls_class']].isna().any().any():
    raise RuntimeError('Incomplete ILS ground-truth fields among WITH_ILS learners.')

# Keanggotaan holdout peserta: 322 train dan 139 test, dengan seed protokol 161600.
FROZEN_PENDIKOM_SPLIT_ROWS = [
    (1008, 'train'),
    (1016, 'train'),
    (1026, 'train'),
    (1029, 'train'),
    (1056, 'train'),
    (1059, 'train'),
    (1077, 'train'),
    (1079, 'train'),
    (1103, 'train'),
    (1147, 'train'),
    (1150, 'train'),
    (1164, 'train'),
    (1170, 'train'),
    (1180, 'train'),
    (1183, 'train'),
    (1185, 'train'),
    (1193, 'train'),
    (1196, 'train'),
    (1214, 'train'),
    (1215, 'train'),
    (1224, 'train'),
    (1238, 'train'),
    (1245, 'train'),
    (1252, 'train'),
    (1256, 'train'),
    (1271, 'train'),
    (1284, 'train'),
    (1285, 'train'),
    (1292, 'train'),
    (1315, 'train'),
    (1325, 'train'),
    (1347, 'train'),
    (1349, 'train'),
    (1355, 'train'),
    (1358, 'train'),
    (1361, 'train'),
    (1386, 'train'),
    (1397, 'train'),
    (1403, 'train'),
    (1409, 'train'),
    (1414, 'train'),
    (1421, 'train'),
    (1424, 'train'),
    (1431, 'train'),
    (1435, 'train'),
    (1440, 'train'),
    (1455, 'train'),
    (1461, 'train'),
    (1462, 'train'),
    (1465, 'train'),
    (1473, 'train'),
    (1482, 'train'),
    (1483, 'train'),
    (1505, 'train'),
    (1528, 'train'),
    (1533, 'train'),
    (1535, 'train'),
    (1547, 'train'),
    (1551, 'train'),
    (1564, 'train'),
    (1567, 'train'),
    (1573, 'train'),
    (1592, 'train'),
    (1626, 'train'),
    (1630, 'train'),
    (1634, 'train'),
    (1638, 'train'),
    (1641, 'train'),
    (1648, 'train'),
    (1652, 'train'),
    (1658, 'train'),
    (1673, 'train'),
    (1674, 'train'),
    (1676, 'train'),
    (1685, 'train'),
    (1686, 'train'),
    (1689, 'train'),
    (1700, 'train'),
    (1706, 'train'),
    (1707, 'train'),
    (1718, 'train'),
    (1721, 'train'),
    (1731, 'train'),
    (1735, 'train'),
    (1749, 'train'),
    (1758, 'train'),
    (1762, 'train'),
    (1769, 'train'),
    (1781, 'train'),
    (1784, 'train'),
    (1787, 'train'),
    (1790, 'train'),
    (1791, 'train'),
    (1797, 'train'),
    (1824, 'train'),
    (1825, 'train'),
    (1835, 'train'),
    (1846, 'train'),
    (1850, 'train'),
    (1863, 'train'),
    (1866, 'train'),
    (1867, 'train'),
    (1878, 'train'),
    (1881, 'train'),
    (1890, 'train'),
    (1893, 'train'),
    (1896, 'train'),
    (1906, 'train'),
    (1915, 'train'),
    (1928, 'train'),
    (1932, 'train'),
    (1936, 'train'),
    (1950, 'train'),
    (1960, 'train'),
    (1966, 'train'),
    (1985, 'train'),
    (1994, 'train'),
    (1996, 'train'),
    (2002, 'train'),
    (2010, 'train'),
    (2029, 'train'),
    (2041, 'train'),
    (2049, 'train'),
    (2051, 'train'),
    (2055, 'train'),
    (2060, 'train'),
    (2066, 'train'),
    (2083, 'train'),
    (2088, 'train'),
    (2089, 'train'),
    (2095, 'train'),
    (2096, 'train'),
    (2099, 'train'),
    (2102, 'train'),
    (2106, 'train'),
    (2110, 'train'),
    (2111, 'train'),
    (2116, 'train'),
    (2142, 'train'),
    (2147, 'train'),
    (2149, 'train'),
    (2156, 'train'),
    (2168, 'train'),
    (2169, 'train'),
    (2176, 'train'),
    (2178, 'train'),
    (2189, 'train'),
    (2194, 'train'),
    (2197, 'train'),
    (2208, 'train'),
    (2215, 'train'),
    (2226, 'train'),
    (2227, 'train'),
    (2230, 'train'),
    (2231, 'train'),
    (2254, 'train'),
    (2258, 'train'),
    (2269, 'train'),
    (2301, 'train'),
    (2308, 'train'),
    (2310, 'train'),
    (2311, 'train'),
    (2315, 'train'),
    (2325, 'train'),
    (2343, 'train'),
    (2351, 'train'),
    (2363, 'train'),
    (2365, 'train'),
    (2370, 'train'),
    (2389, 'train'),
    (2405, 'train'),
    (2429, 'train'),
    (2436, 'train'),
    (2461, 'train'),
    (2468, 'train'),
    (2482, 'train'),
    (2493, 'train'),
    (2500, 'train'),
    (2501, 'train'),
    (2503, 'train'),
    (2509, 'train'),
    (2519, 'train'),
    (2531, 'train'),
    (2545, 'train'),
    (2571, 'train'),
    (2575, 'train'),
    (2580, 'train'),
    (2581, 'train'),
    (2583, 'train'),
    (2595, 'train'),
    (2606, 'train'),
    (2609, 'train'),
    (2617, 'train'),
    (2637, 'train'),
    (2638, 'train'),
    (2639, 'train'),
    (2648, 'train'),
    (2652, 'train'),
    (2672, 'train'),
    (2674, 'train'),
    (2677, 'train'),
    (2682, 'train'),
    (2695, 'train'),
    (2707, 'train'),
    (2715, 'train'),
    (2719, 'train'),
    (2728, 'train'),
    (2738, 'train'),
    (2739, 'train'),
    (2742, 'train'),
    (2764, 'train'),
    (2773, 'train'),
    (2785, 'train'),
    (2786, 'train'),
    (2790, 'train'),
    (2793, 'train'),
    (2796, 'train'),
    (2803, 'train'),
    (2829, 'train'),
    (2833, 'train'),
    (2838, 'train'),
    (2864, 'train'),
    (2867, 'train'),
    (2874, 'train'),
    (2885, 'train'),
    (2888, 'train'),
    (2892, 'train'),
    (2896, 'train'),
    (2910, 'train'),
    (2929, 'train'),
    (2946, 'train'),
    (2951, 'train'),
    (2962, 'train'),
    (2969, 'train'),
    (2971, 'train'),
    (2975, 'train'),
    (2979, 'train'),
    (2986, 'train'),
    (2998, 'train'),
    (3004, 'train'),
    (3046, 'train'),
    (3048, 'train'),
    (3054, 'train'),
    (3062, 'train'),
    (3088, 'train'),
    (3095, 'train'),
    (3100, 'train'),
    (3103, 'train'),
    (3111, 'train'),
    (3114, 'train'),
    (3124, 'train'),
    (3135, 'train'),
    (3142, 'train'),
    (3148, 'train'),
    (3150, 'train'),
    (3157, 'train'),
    (3177, 'train'),
    (3182, 'train'),
    (3183, 'train'),
    (3188, 'train'),
    (3195, 'train'),
    (3217, 'train'),
    (3221, 'train'),
    (3224, 'train'),
    (3231, 'train'),
    (3253, 'train'),
    (3262, 'train'),
    (3263, 'train'),
    (3266, 'train'),
    (3271, 'train'),
    (3279, 'train'),
    (3287, 'train'),
    (3291, 'train'),
    (3295, 'train'),
    (3308, 'train'),
    (3320, 'train'),
    (3329, 'train'),
    (3330, 'train'),
    (3333, 'train'),
    (3343, 'train'),
    (3370, 'train'),
    (3374, 'train'),
    (3381, 'train'),
    (3385, 'train'),
    (3407, 'train'),
    (3433, 'train'),
    (3438, 'train'),
    (3456, 'train'),
    (3466, 'train'),
    (3472, 'train'),
    (3476, 'train'),
    (3484, 'train'),
    (3488, 'train'),
    (3492, 'train'),
    (3500, 'train'),
    (3511, 'train'),
    (3519, 'train'),
    (3530, 'train'),
    (3533, 'train'),
    (3541, 'train'),
    (3557, 'train'),
    (3558, 'train'),
    (3562, 'train'),
    (3568, 'train'),
    (3571, 'train'),
    (3579, 'train'),
    (3588, 'train'),
    (3592, 'train'),
    (3608, 'train'),
    (3609, 'train'),
    (3621, 'train'),
    (3630, 'train'),
    (3634, 'train'),
    (3654, 'train'),
    (3658, 'train'),
    (3672, 'train'),
    (3694, 'train'),
    (3706, 'train'),
    (3718, 'train'),
    (3726, 'train'),
    (3744, 'train'),
    (3761, 'train'),
    (1015, 'test'),
    (1023, 'test'),
    (1055, 'test'),
    (1082, 'test'),
    (1107, 'test'),
    (1109, 'test'),
    (1113, 'test'),
    (1115, 'test'),
    (1146, 'test'),
    (1191, 'test'),
    (1204, 'test'),
    (1208, 'test'),
    (1249, 'test'),
    (1273, 'test'),
    (1277, 'test'),
    (1301, 'test'),
    (1314, 'test'),
    (1316, 'test'),
    (1320, 'test'),
    (1339, 'test'),
    (1340, 'test'),
    (1368, 'test'),
    (1383, 'test'),
    (1388, 'test'),
    (1407, 'test'),
    (1430, 'test'),
    (1451, 'test'),
    (1452, 'test'),
    (1475, 'test'),
    (1489, 'test'),
    (1498, 'test'),
    (1508, 'test'),
    (1530, 'test'),
    (1544, 'test'),
    (1562, 'test'),
    (1565, 'test'),
    (1588, 'test'),
    (1657, 'test'),
    (1661, 'test'),
    (1681, 'test'),
    (1712, 'test'),
    (1729, 'test'),
    (1745, 'test'),
    (1752, 'test'),
    (1757, 'test'),
    (1766, 'test'),
    (1809, 'test'),
    (1831, 'test'),
    (1838, 'test'),
    (1845, 'test'),
    (1852, 'test'),
    (1884, 'test'),
    (1934, 'test'),
    (1956, 'test'),
    (1988, 'test'),
    (2000, 'test'),
    (2067, 'test'),
    (2077, 'test'),
    (2084, 'test'),
    (2103, 'test'),
    (2114, 'test'),
    (2126, 'test'),
    (2132, 'test'),
    (2136, 'test'),
    (2160, 'test'),
    (2199, 'test'),
    (2223, 'test'),
    (2242, 'test'),
    (2247, 'test'),
    (2272, 'test'),
    (2275, 'test'),
    (2291, 'test'),
    (2292, 'test'),
    (2299, 'test'),
    (2335, 'test'),
    (2402, 'test'),
    (2415, 'test'),
    (2416, 'test'),
    (2420, 'test'),
    (2472, 'test'),
    (2492, 'test'),
    (2495, 'test'),
    (2577, 'test'),
    (2620, 'test'),
    (2628, 'test'),
    (2632, 'test'),
    (2666, 'test'),
    (2692, 'test'),
    (2704, 'test'),
    (2745, 'test'),
    (2758, 'test'),
    (2789, 'test'),
    (2817, 'test'),
    (2823, 'test'),
    (2881, 'test'),
    (2918, 'test'),
    (2930, 'test'),
    (2932, 'test'),
    (2940, 'test'),
    (2957, 'test'),
    (3016, 'test'),
    (3119, 'test'),
    (3126, 'test'),
    (3144, 'test'),
    (3160, 'test'),
    (3162, 'test'),
    (3165, 'test'),
    (3166, 'test'),
    (3190, 'test'),
    (3196, 'test'),
    (3228, 'test'),
    (3238, 'test'),
    (3240, 'test'),
    (3267, 'test'),
    (3284, 'test'),
    (3303, 'test'),
    (3311, 'test'),
    (3376, 'test'),
    (3392, 'test'),
    (3412, 'test'),
    (3423, 'test'),
    (3447, 'test'),
    (3457, 'test'),
    (3459, 'test'),
    (3463, 'test'),
    (3469, 'test'),
    (3507, 'test'),
    (3548, 'test'),
    (3567, 'test'),
    (3585, 'test'),
    (3614, 'test'),
    (3647, 'test'),
    (3668, 'test'),
    (3678, 'test'),
    (3689, 'test'),
    (3690, 'test'),
    (3720, 'test'),
    (3728, 'test'),
    (3742, 'test')
]

split = pd.DataFrame(FROZEN_PENDIKOM_SPLIT_ROWS, columns=['userid','split'])
if not {'userid','split'}.issubset(split.columns):
    raise RuntimeError('Frozen split must contain userid and split columns.')

split['userid'] = pd.to_numeric(split['userid'], errors='raise').astype(int)
split['split'] = split['split'].astype(str).str.strip().str.lower()

if split['userid'].duplicated().any():
    raise RuntimeError('Frozen split contains duplicate userid values.')
if set(split['split']) != {'train','test'}:
    raise RuntimeError(f'Unexpected split labels: {sorted(split["split"].unique())}')
if set(split['userid']) != set(pendikom_ils['userid']):
    raise RuntimeError('Frozen split and WITH_ILS cohort do not contain the same users.')

train_ids = split.loc[split['split'].eq('train'), 'userid'].tolist()
test_ids = split.loc[split['split'].eq('test'), 'userid'].tolist()

ils_train = (
    pd.DataFrame({'userid': train_ids})
    .merge(pendikom_ils, on='userid', how='left', validate='one_to_one')
)
ils_test = (
    pd.DataFrame({'userid': test_ids})
    .merge(pendikom_ils, on='userid', how='left', validate='one_to_one')
)

if len(ils_train) != 322 or len(ils_test) != 139:
    raise RuntimeError(
        f'Expected 322/139 train/test rows, got {len(ils_train)}/{len(ils_test)}.'
    )



## Pembentukan fitur numerik dan kategorikal


In [ ]:

# ============================================================
# Train-fitted learner-level preprocessing
# 19 numeric indicators -> categorical 1/2/3
# ============================================================

def _fit_q75_reference(train_frame, seconds_col, opportunity_col=None):
    cols = ['courseid', seconds_col] + ([opportunity_col] if opportunity_col else [])
    t = train_frame[cols].copy()
    if opportunity_col is not None:
        t = t[t[opportunity_col].gt(0)]
    positive = t[t[seconds_col].gt(0)]

    by_course = positive.groupby('courseid')[seconds_col].quantile(0.75)
    global_q75 = float(positive[seconds_col].quantile(0.75)) if len(positive) else 1.0
    if not np.isfinite(global_q75) or global_q75 <= 0:
        global_q75 = 1.0
    return by_course, global_q75


def _ratio_pct(num, den):
    n = pd.to_numeric(num, errors='coerce').fillna(0).to_numpy(float)
    d = pd.to_numeric(den, errors='coerce').fillna(0).to_numpy(float)
    out = np.zeros(len(n), dtype=float)
    mask = d > 0
    out[mask] = 100.0 * n[mask] / d[mask]
    return out


stay_refs = {
    'content': _fit_q75_reference(
        ils_train, 'content_stay_seconds', 'content_available'
    ),
    'exercise': _fit_q75_reference(
        ils_train, 'exercise_stay_seconds', 'assign_available'
    ),
    'overview': _fit_q75_reference(
        ils_train, 'overview_stay_seconds', None
    ),
}


def build_pendikom_numeric(frame):
    b = frame
    meta = {
        'userid': b['userid'].astype(int),
        'courseid': b['courseid'].astype(int),
        'total_activity': pd.to_numeric(b['total_activity'], errors='raise').astype(int),
        'active_days': pd.to_numeric(b['active_days'], errors='raise').astype(int),
    }
    if 'course_fullname' in b.columns:
        meta['course_fullname'] = b['course_fullname'].astype(str)

    out = pd.DataFrame(meta)
    out['content_visit'] = _ratio_pct(
        b['content_visit_evidence_count'], b['content_available']
    )

    ref_by_course, ref_global = stay_refs['content']
    ref = b['courseid'].map(ref_by_course).fillna(ref_global).to_numpy(float)
    sec = pd.to_numeric(b['content_stay_seconds'], errors='coerce').fillna(0).to_numpy(float)
    out['content_stay'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)

    out['selfass_visit'] = _ratio_pct(
        b['question_attempt_count'], b['question_available']
    )
    out['selfass_twice_wrong'] = _ratio_pct(
        b['twice_wrong_question_count'], b['repeated_question_count']
    )
    out['exercise_visit'] = _ratio_pct(
        b['submitted_assignment_count'], b['assign_available']
    )

    ref_by_course, ref_global = stay_refs['exercise']
    ref = b['courseid'].map(ref_by_course).fillna(ref_global).to_numpy(float)
    sec = pd.to_numeric(b['exercise_stay_seconds'], errors='coerce').fillna(0).to_numpy(float)
    out['exercise_stay'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)

    for tag in ['detail','overview','facts','concepts','graphics','text','interpret','develop']:
        numerator = (
            'interpret_question_score_sum'
            if tag == 'interpret'
            else f'ques_{tag}_fraction_sum'
        )
        out[f'ques_{tag}'] = _ratio_pct(
            b[numerator], b[f'ques_{tag}_attempted_count']
        )

    out['quiz_revisions'] = _ratio_pct(
        b['revised_question_attempt_count'], b['question_attempt_count']
    )

    events = pd.to_numeric(
        b['quiz_results_event_count'], errors='coerce'
    ).fillna(0).to_numpy(float)
    seconds = pd.to_numeric(
        b['quiz_results_stay_seconds_sum'], errors='coerce'
    ).fillna(0).to_numpy(float)
    q = np.zeros(len(b), dtype=float)
    mask = events > 0
    q[mask] = seconds[mask] / events[mask]
    out['quiz_stay_results'] = q

    module_views = np.maximum(
        pd.to_numeric(b['module_view_count'], errors='coerce').fillna(0).to_numpy(float),
        1.0,
    )
    out['navigation_skip'] = (
        100.0 *
        pd.to_numeric(b['navigation_skip_count'], errors='coerce').fillna(0).to_numpy(float)
        / module_views
    )
    out['navigation_overview_visit'] = (
        100.0 *
        pd.to_numeric(b['overview_visit_count'], errors='coerce').fillna(0).to_numpy(float)
        / module_views
    )

    ref_by_course, ref_global = stay_refs['overview']
    ref = b['courseid'].map(ref_by_course).fillna(ref_global).to_numpy(float)
    sec = pd.to_numeric(b['overview_stay_seconds'], errors='coerce').fillna(0).to_numpy(float)
    out['navigation_overview_stay'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)

    return out


def categorize_pendikom(numeric_frame):
    meta_cols = [
        c for c in ['userid','courseid','course_fullname','total_activity','active_days']
        if c in numeric_frame.columns
    ]
    out = numeric_frame[meta_cols].copy()
    for feature in PENDIKOM_FEATURES:
        low, high = PENDIKOM_THRESHOLDS[feature]
        v = pd.to_numeric(numeric_frame[feature], errors='coerce').fillna(0).to_numpy(float)
        out[feature] = np.where(
            v < low, 1, np.where(v < high, 2, 3)
        ).astype(np.int8)
    return out


gt_train_numeric = build_pendikom_numeric(ils_train)
gt_test_numeric = build_pendikom_numeric(ils_test)
gt_train = categorize_pendikom(gt_train_numeric)
gt_test = categorize_pendikom(gt_test_numeric)

constant_features = [
    f for f in PENDIKOM_FEATURES
    if gt_train[f].nunique(dropna=False) == 1
]
features = [f for f in PENDIKOM_FEATURES if f not in constant_features]



In [ ]:
FITUR_MODEL = ['content_visit', 'content_stay', 'selfass_visit', 'selfass_twice_wrong', 'exercise_visit', 'exercise_stay', 'ques_detail', 'ques_overview', 'ques_facts', 'ques_concepts', 'ques_graphics', 'ques_text', 'ques_interpret', 'ques_develop', 'quiz_revisions', 'quiz_stay_results', 'navigation_overview_visit', 'navigation_overview_stay']
if features != FITUR_MODEL:
    raise RuntimeError(f'Daftar fitur berbeda dari protokol penelitian: {features}')

def gabungkan_hasil(train_frame,test_frame):
    bagian=[]
    for label,frame in [('train',train_frame),('test',test_frame)]:
        f=frame[['userid','courseid']+features].copy()
        f['split']=label
        bagian.append(f[['userid','courseid','split']+features])
    return pd.concat(bagian,ignore_index=True)

numeric=gabungkan_hasil(gt_train_numeric,gt_test_numeric)
categorical=gabungkan_hasil(gt_train,gt_test)
assert len(numeric)==len(categorical)==461
assert numeric['userid'].is_unique and categorical['userid'].is_unique
assert numeric[['userid','courseid','split']].equals(categorical[['userid','courseid','split']])
assert categorical[features].isin([1,2,3]).all().all()

# Ekspor dua dataset behavioral.
payload={
    '01_numeric_behavioral_pendikom.csv':numeric.to_csv(index=False,lineterminator='\n').encode('utf-8'),
    '02_categorical_behavioral_pendikom.csv':categorical.to_csv(index=False,lineterminator='\n').encode('utf-8'),
}
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for filename,data in payload.items():
    (OUTPUT_DIR/filename).write_bytes(data)
print('Selesai: dua CSV, masing-masing 461 peserta dan 18 fitur.')
print('Kolom identitas/metadata: userid, courseid, split. Split: 322 train / 139 test.')
print('Fitur yang dikeluarkan:',constant_features)
print('Lokasi:',OUTPUT_DIR)
